In [ ]:
import matplotlib.pyplot as plt


In [ ]:
# =============================================================================
# PROBLEM DEFINITION
# =============================================================================

TIME_HORIZON = 24
ENERGY_PRICE = {
    0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8,          # off-peak
    6: 18, 7: 18, 8: 18, 9: 18,                  # morning peak
    10: 12, 11: 12, 12: 12, 13: 12, 14: 12, 15: 12, 16: 12,
    17: 22, 18: 22, 19: 22, 20: 22, 21: 22,      # evening peak
    22: 10, 23: 10                               # late night
}

@dataclass
class Server:
    id: str
    server_type: str
    power_kw: float

@dataclass
class Operation:
    server_type: str
    duration: int

@dataclass
class Job:
    id: str
    description: str
    priority: str
    deadline: int
    delay_weight: float
    operations: List[Operation]

@dataclass
class Instance:
    servers: Dict[str, Server]
    jobs: Dict[str, Job]
    energy_price: Dict[int, float]
    time_horizon: int
    weights: Dict[str, float]

Schedule = Dict[str, List[Dict[str, Any]]]

def create_instance() -> Instance:
    servers = {
        "S1": Server("S1", "CPU", 2.0),
        "S2": Server("S2", "CPU", 2.0),
        "S3": Server("S3", "GPU", 4.5),
        "S4": Server("S4", "Storage", 1.2),
    }
    jobs = {
        "J1": Job("J1", "Emergency response analytics", "HIGH", 8, 10.0, [
            Operation("Storage", 1), Operation("CPU", 2), Operation("GPU", 1)
        ]),
        "J2": Job("J2", "Disaster damage estimation", "HIGH", 10, 8.0, [
            Operation("Storage", 1), Operation("GPU", 2), Operation("CPU", 1)
        ]),
        "J3": Job("J3", "Tax data aggregation", "MEDIUM", 18, 4.0, [
            Operation("CPU", 2), Operation("Storage", 1), Operation("CPU", 1)
        ]),
        "J4": Job("J4", "Census preprocessing", "MEDIUM", 20, 3.0, [
            Operation("Storage", 1), Operation("CPU", 2)
        ]),
        "J5": Job("J5", "Open data batch update", "LOW", 22, 1.0, [
            Operation("CPU", 1), Operation("Storage", 1)
        ]),
        "J6": Job("J6", "Archive and backup", "LOW", 24, 0.5, [
            Operation("Storage", 2), Operation("CPU", 2)
        ]),
    }
    return Instance(servers, jobs, ENERGY_PRICE, TIME_HORIZON, 
                   {"energy_cost": 1.0, "delay_penalty": 5.0})

INSTANCE = create_instance()
print(f"Instance: {len(INSTANCE.jobs)} jobs, {len(INSTANCE.servers)} servers")

In [ ]:
# =============================================================================
# FITNESS EVALUATION
# =============================================================================

@dataclass
class Fitness:
    score: float
    energy_cost: float
    delay_penalty: float
    violations: List[str]
    job_completion_times: Dict[str, int]
    
    def is_valid(self) -> bool:
        return len(self.violations) == 0


def evaluate(inst: Instance, schedule: Schedule) -> Fitness:
    server_usage = {s: [] for s in inst.servers}
    job_completion_times = {}
    total_energy_cost = 0.0
    total_delay_penalty = 0.0
    violations = []

    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        if len(ops) != len(job.operations):
            violations.append(f"{job_id}: wrong number of operations")
            continue

        prev_end = None
        for idx, op in enumerate(ops):
            server_id = op["server"]
            start = op["start"]
            duration = job.operations[idx].duration
            end = start + duration

            if prev_end is not None and start < prev_end:
                violations.append(f"{job_id}: op {idx} starts before previous ends")
            prev_end = end

            required_type = job.operations[idx].server_type
            actual_type = inst.servers[server_id].server_type
            if required_type != actual_type:
                violations.append(f"{job_id}: op {idx} requires {required_type}, got {actual_type}")

            server_usage[server_id].append((start, end, job_id, idx))
            power = inst.servers[server_id].power_kw
            for t in range(start, end):
                total_energy_cost += power * inst.energy_price[t % inst.time_horizon]

        completion = prev_end if prev_end else 0
        job_completion_times[job_id] = completion
        delay = max(0, completion - job.deadline)
        total_delay_penalty += delay * job.delay_weight

    for server_id, intervals in server_usage.items():
        intervals.sort()
        for i in range(len(intervals) - 1):
            _, end1, job1, _ = intervals[i]
            start2, _, job2, _ = intervals[i + 1]
            if start2 < end1:
                violations.append(f"{server_id}: overlap between {job1} and {job2}")

    score = (inst.weights["energy_cost"] * total_energy_cost +
             inst.weights["delay_penalty"] * total_delay_penalty +
             len(violations) * 10000)

    return Fitness(round(score, 2), round(total_energy_cost, 2),
                   round(total_delay_penalty, 2), violations, job_completion_times)

In [ ]:
# EVOLVE-BLOCK-START

def construct_schedule(inst: Instance) -> Schedule:
    """
    Constructs a schedule for the job-shop scheduling problem.
    This function is evolved by ShinkaEvolve/AlphaEvolve.
    
    Args:
        inst: Problem instance with servers, jobs, energy prices
    
    Returns:
        Schedule: Dict mapping job_id -> list of {"server": str, "start": int}
    """
    def get_servers_by_type(server_type: str) -> List[str]:
        return [s.id for s in inst.servers.values() if s.server_type == server_type]
    
    def op_energy_cost(server_id: str, start: int, duration: int) -> float:
        power = inst.servers[server_id].power_kw
        return sum(power * inst.energy_price[t % inst.time_horizon]
                   for t in range(start, start + duration))
    
    # --- Initial solution: Priority-based with energy awareness ---
    priority_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    sorted_jobs = sorted(inst.jobs.keys(),
                        key=lambda j: (priority_order[inst.jobs[j].priority],
                                      inst.jobs[j].deadline, j))
    
    server_available = {s: 0 for s in inst.servers}
    job_available = {j: 0 for j in inst.jobs}
    schedule = {}
    
    for job_id in sorted_jobs:
        job = inst.jobs[job_id]
        schedule[job_id] = []
        
        for op in job.operations:
            compatible = get_servers_by_type(op.server_type)
            best_choice, best_score = None, float("inf")
            
            for server_id in compatible:
                earliest = max(job_available[job_id], server_available[server_id])
                
                # Try a few start times to find energy-efficient slot
                for start in range(earliest, min(earliest + 6, inst.time_horizon - op.duration + 1)):
                    energy = op_energy_cost(server_id, start, op.duration)
                    # Balance energy cost with completion time
                    delay_risk = max(0, start + op.duration - job.deadline) * job.delay_weight
                    score = energy + delay_risk * 2
                    
                    if score < best_score:
                        best_score = score
                        best_choice = (server_id, start)
            
            if best_choice is None:
                server_id = compatible[0]
                start = max(job_available[job_id], server_available[server_id])
            else:
                server_id, start = best_choice
            
            schedule[job_id].append({"server": server_id, "start": start})
            finish = start + op.duration
            job_available[job_id] = finish
            server_available[server_id] = finish
    
    # --- Local search improvement ---
    for _ in range(50):
        improved = False
        current_fit = evaluate(inst, schedule)
        
        for job_id in inst.jobs:
            job = inst.jobs[job_id]
            for op_idx in range(len(job.operations)):
                op = job.operations[op_idx]
                compatible = get_servers_by_type(op.server_type)
                
                # Try swapping to different server or shifting time
                for new_server in compatible:
                    for delta in range(-3, 4):
                        new_start = max(0, schedule[job_id][op_idx]["start"] + delta)
                        
                        # Create candidate schedule
                        candidate = copy.deepcopy(schedule)
                        candidate[job_id][op_idx] = {"server": new_server, "start": new_start}
                        
                        new_fit = evaluate(inst, candidate)
                        if new_fit.score < current_fit.score:
                            schedule = candidate
                            current_fit = new_fit
                            improved = True
                            break
                    if improved:
                        break
                if improved:
                    break
            if improved:
                break
        
        if not improved:
            break
    
    return schedule

# EVOLVE-BLOCK-END


def run_scheduling():
    """Run the schedule constructor and return results."""
    schedule = construct_schedule(INSTANCE)
    fitness = evaluate(INSTANCE, schedule)
    return schedule, fitness


np.random.seed(42)
schedule, fitness = run_scheduling()
print(f"Score: {fitness.score}")
print(f"Energy Cost: {fitness.energy_cost}")
print(f"Delay Penalty: {fitness.delay_penalty}")
print(f"Valid: {fitness.is_valid()}")

In [ ]:
# =============================================================================
# VALIDATION
# =============================================================================

def validate_schedule(
    run_output: Tuple[Schedule, Fitness],
    inst: Instance = INSTANCE,
    atol: float = 1e-7
) -> Tuple[bool, Optional[str]]:
    """
    Validates schedule results based on the output of 'run_scheduling'.
    
    Args:
        run_output: Tuple (schedule, fitness) from run_scheduling.
        inst: Problem instance.
        atol: Absolute tolerance for numerical checks.
    
    Returns:
        (is_valid: bool, error_message: Optional[str])
    """
    schedule, fitness = run_output
    
    # Check all jobs are scheduled
    if set(schedule.keys()) != set(inst.jobs.keys()):
        missing = set(inst.jobs.keys()) - set(schedule.keys())
        extra = set(schedule.keys()) - set(inst.jobs.keys())
        return False, f"Job mismatch. Missing: {missing}, Extra: {extra}"
    
    # Check operation counts
    for job_id, ops in schedule.items():
        expected = len(inst.jobs[job_id].operations)
        if len(ops) != expected:
            return False, f"{job_id}: expected {expected} ops, got {len(ops)}"
    
    # Check server type compatibility
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        for idx, op_sched in enumerate(ops):
            required = job.operations[idx].server_type
            actual = inst.servers[op_sched["server"]].server_type
            if required != actual:
                return False, f"{job_id} op {idx}: requires {required}, assigned {actual}"
    
    # Check job precedence constraints
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        prev_end = 0
        for idx, op_sched in enumerate(ops):
            start = op_sched["start"]
            duration = job.operations[idx].duration
            if start < prev_end:
                return False, f"{job_id} op {idx}: starts at {start} before prev ends at {prev_end}"
            prev_end = start + duration
    
    # Check server capacity (no overlaps)
    server_intervals = {s: [] for s in inst.servers}
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        for idx, op_sched in enumerate(ops):
            server = op_sched["server"]
            start = op_sched["start"]
            end = start + job.operations[idx].duration
            server_intervals[server].append((start, end, job_id))
    
    for server, intervals in server_intervals.items():
        intervals.sort()
        for i in range(len(intervals) - 1):
            _, end1, job1 = intervals[i]
            start2, _, job2 = intervals[i + 1]
            if start2 < end1:
                return False, f"{server}: overlap between {job1} (ends {end1}) and {job2} (starts {start2})"
    
    # Check non-negative start times
    for job_id, ops in schedule.items():
        for idx, op_sched in enumerate(ops):
            if op_sched["start"] < 0:
                return False, f"{job_id} op {idx}: negative start time {op_sched['start']}"
    
    # Check fitness violations match
    if fitness.violations:
        return False, f"Fitness has violations: {fitness.violations}"
    
    return True, None


# Run validation
is_valid, error = validate_schedule((schedule, fitness))
print(f"Valid: {is_valid}")
if error:
    print(f"Error: {error}")

In [ ]:
# =============================================================================
# VISUALIZATION - Gantt Chart
# =============================================================================

def visualize_schedule(schedule: Schedule, inst: Instance, fitness: Fitness):
    """
    Visualize the schedule as a Gantt chart with two views:
    1. Server view (which jobs run on each server)
    2. Job view (timeline of each job's operations)
    """
    # Color palette for jobs
    job_colors = {
        "J1": "#e74c3c",  # red (HIGH)
        "J2": "#e67e22",  # orange (HIGH)
        "J3": "#3498db",  # blue (MEDIUM)
        "J4": "#9b59b6",  # purple (MEDIUM)
        "J5": "#2ecc71",  # green (LOW)
        "J6": "#1abc9c",  # teal (LOW)
    }
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # --- Server View (Top) ---
    ax1 = axes[0]
    servers = list(inst.servers.keys())
    server_y = {s: i for i, s in enumerate(servers)}
    
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        for idx, op_sched in enumerate(ops):
            server = op_sched["server"]
            start = op_sched["start"]
            duration = job.operations[idx].duration
            y = server_y[server]
            
            rect = Rectangle((start, y - 0.4), duration, 0.8,
                            facecolor=job_colors[job_id], edgecolor='black',
                            linewidth=1.5, alpha=0.8)
            ax1.add_patch(rect)
            ax1.text(start + duration/2, y, f"{job_id}\nop{idx}",
                    ha='center', va='center', fontsize=8, fontweight='bold')
    
    # Draw energy price background
    for t in range(inst.time_horizon):
        price = inst.energy_price[t]
        alpha = 0.1 + 0.15 * (price - 8) / 14  # Normalize to 0.1-0.25
        ax1.axvspan(t, t+1, alpha=alpha, color='yellow', zorder=0)
    
    ax1.set_xlim(0, inst.time_horizon)
    ax1.set_ylim(-0.5, len(servers) - 0.5)
    ax1.set_yticks(range(len(servers)))
    ax1.set_yticklabels([f"{s} ({inst.servers[s].server_type})" for s in servers])
    ax1.set_xlabel("Time (hours)")
    ax1.set_title(f"Server View | Score: {fitness.score} | Energy: {fitness.energy_cost} | Delay: {fitness.delay_penalty}")
    ax1.grid(True, axis='x', alpha=0.3)
    ax1.set_xticks(range(0, inst.time_horizon + 1, 2))
    
    # --- Job View (Bottom) ---
    ax2 = axes[1]
    jobs = list(inst.jobs.keys())
    job_y = {j: i for i, j in enumerate(jobs)}
    
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        y = job_y[job_id]
        
        for idx, op_sched in enumerate(ops):
            server = op_sched["server"]
            start = op_sched["start"]
            duration = job.operations[idx].duration
            
            rect = Rectangle((start, y - 0.4), duration, 0.8,
                            facecolor=job_colors[job_id], edgecolor='black',
                            linewidth=1.5, alpha=0.8)
            ax2.add_patch(rect)
            ax2.text(start + duration/2, y, f"{server}",
                    ha='center', va='center', fontsize=8, fontweight='bold')
        
        # Draw deadline marker
        ax2.axvline(x=job.deadline, ymin=(y - 0.5 + 0.5) / len(jobs),
                   ymax=(y + 0.5 + 0.5) / len(jobs),
                   color='red', linestyle='--', linewidth=2, alpha=0.7)
        
        # Mark completion time
        completion = fitness.job_completion_times.get(job_id, 0)
        marker = '✓' if completion <= job.deadline else '✗'
        color = 'green' if completion <= job.deadline else 'red'
        ax2.text(completion + 0.2, y, marker, fontsize=12, color=color, va='center')
    
    ax2.set_xlim(0, inst.time_horizon)
    ax2.set_ylim(-0.5, len(jobs) - 0.5)
    ax2.set_yticks(range(len(jobs)))
    ax2.set_yticklabels([f"{j} ({inst.jobs[j].priority})" for j in jobs])
    ax2.set_xlabel("Time (hours)")
    ax2.set_title("Job View (red dashed = deadline)")
    ax2.grid(True, axis='x', alpha=0.3)
    ax2.set_xticks(range(0, inst.time_horizon + 1, 2))
    
    plt.tight_layout()
    return fig, axes


fig, axes = visualize_schedule(schedule, INSTANCE, fitness)
fig.savefig("schedule_gantt.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# =============================================================================
# ENERGY PRICE VISUALIZATION
# =============================================================================

def visualize_energy_usage(schedule: Schedule, inst: Instance):
    """Visualize energy consumption over time with price overlay."""
    fig, ax1 = plt.subplots(figsize=(12, 5))
    
    # Calculate energy consumption per hour
    energy_by_hour = [0.0] * inst.time_horizon
    for job_id, ops in schedule.items():
        job = inst.jobs[job_id]
        for idx, op_sched in enumerate(ops):
            server = op_sched["server"]
            start = op_sched["start"]
            duration = job.operations[idx].duration
            power = inst.servers[server].power_kw
            for t in range(start, start + duration):
                if t < inst.time_horizon:
                    energy_by_hour[t] += power
    
    # Plot energy consumption
    hours = list(range(inst.time_horizon))
    ax1.bar(hours, energy_by_hour, color='steelblue', alpha=0.7, label='Power (kW)')
    ax1.set_xlabel('Hour')
    ax1.set_ylabel('Power Consumption (kW)', color='steelblue')
    ax1.tick_params(axis='y', labelcolor='steelblue')
    
    # Plot energy prices on secondary axis
    ax2 = ax1.twinx()
    prices = [inst.energy_price[t] for t in hours]
    ax2.plot(hours, prices, color='red', linewidth=2, marker='o', markersize=4, label='Price (¥/kWh)')
    ax2.set_ylabel('Energy Price (¥/kWh)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    
    # Calculate cost per hour
    cost_by_hour = [energy_by_hour[t] * inst.energy_price[t] for t in hours]
    total_cost = sum(cost_by_hour)
    
    ax1.set_title(f'Energy Consumption vs Price | Total Cost: ¥{total_cost:.2f}')
    ax1.set_xticks(range(0, inst.time_horizon, 2))
    
    # Add legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    
    plt.tight_layout()
    return fig, (ax1, ax2)


fig, axes = visualize_energy_usage(schedule, INSTANCE)
fig.savefig("energy_usage.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# =============================================================================
# SCHEDULE SUMMARY
# =============================================================================

def print_schedule_summary(schedule: Schedule, inst: Instance, fitness: Fitness):
    """Print a detailed summary of the schedule."""
    print("=" * 60)
    print("SCHEDULE SUMMARY")
    print("=" * 60)
    print(f"\nObjective Score: {fitness.score}")
    print(f"  - Energy Cost: ¥{fitness.energy_cost}")
    print(f"  - Delay Penalty: {fitness.delay_penalty}")
    print(f"  - Valid: {fitness.is_valid()}")
    
    print("\n" + "-" * 60)
    print("JOB DETAILS")
    print("-" * 60)
    
    for job_id in sorted(schedule.keys()):
        job = inst.jobs[job_id]
        completion = fitness.job_completion_times.get(job_id, 0)
        delay = max(0, completion - job.deadline)
        status = "✓ ON TIME" if delay == 0 else f"✗ LATE by {delay}h"
        
        print(f"\n{job_id} ({job.priority}): {job.description}")
        print(f"  Deadline: {job.deadline}h | Completed: {completion}h | {status}")
        print(f"  Operations:")
        
        for idx, op_sched in enumerate(schedule[job_id]):
            op = job.operations[idx]
            server = op_sched["server"]
            start = op_sched["start"]
            end = start + op.duration
            print(f"    [{idx}] {op.server_type} on {server}: t={start}-{end}")


print_schedule_summary(schedule, INSTANCE, fitness)